In [ ]:
#| hide
from fewsats.core import *

# fewsats

> Payments for AI agents - Python


This library enables AI agents to handle real-world payments autonomously. It provides a simple interface for AI systems to manage payment methods, process transactions, and interact with L402 paywalls.

The library provides two main interfaces:
1. Direct payment handling through the Python SDK
2. AI-native tools through `as_tools()` for autonomous agents


## Install

Install latest from the GitHub [repository][repo]:

```sh
$ pip install git+https://github.com/Fewsats/fewsats-python.git
```

or from [pypi][pypi]


```sh
$ pip install fewsats
```


[repo]: https://github.com/Fewsats/fewsats-python
[docs]: https://Fewsats.github.io/fewsats-python/
[pypi]: https://pypi.org/project/fewsats-python/
[conda]: https://anaconda.org/Fewsats/fewsats-python

## Getting Started

The library provides a `Client` class to handle payments. You can use handle them manually or use the `as_tools()` method to create tools for autonomous agents.

In [ ]:
from fewsats.core import * 

In [ ]:
fs = Client()
fs.payment_methods()

[{'id': 5,
  'last4': '4242',
  'brand': 'Visa',
  'exp_month': 12,
  'exp_year': 2034,
  'is_default': True}]

In [ ]:
fs.balance()

[{'id': 15, 'balance': 6471, 'currency': 'usd'}]

In [ ]:
fs.me()

{'name': 'Fewsats',
 'last_name': 'Tester',
 'email': 'test@fewsats.com',
 'billing_info': None,
 'id': 15,
 'created_at': '2024-12-18T18:19:00.531Z'}

The `pay` method uses the information returned by a [L402 Protocol](https://github.com/l402-protocol/l402?tab=readme-ov-file#402-response-format) `402 Payment Required` response to submit a payment.

In [ ]:
# Example offer from stock.l402.org
ofs = {
   "offers":[
      {
         "amount":1,
         "balance":1,
         "currency":"USD",
         "description":"Purchase 1 credit for API access",
         "offer_id":"offer_c668e0c0",
         "payment_methods":[
            "lightning"
         ],
         "title":"1 Credit Package",
         "type":"top-up"
      }
   ],
   "payment_context_token":"edb53dec-28f5-4cbb-924a-20e9003c20e1",
   "payment_request_url":"https://stock.l402.org/l402/payment-request",
   "terms_url":"https://link-to-terms.com",
   "version":"0.2.1"
}

In [ ]:
fs.pay(ofs['payment_request_url'], ofs['payment_context_token'], **ofs['offers'][0], pm='lightning')

{'id': 240,
 'created_at': '2024-12-26T10:24:03.486Z',
 'status': 'success',
 'payment_request_url': 'https://stock.l402.org/l402/payment-request',
 'payment_context_token': 'edb53dec-28f5-4cbb-924a-20e9003c20e1',
 'invoice': 'lnbc100n1pnk6tkrpp5pkshrxfyvxuqphwxvax8dfaemhh287lnefrjr68cy494fppe5l0sdq6xysyxun9v35hggzsv93kkct8v5cqzpgxqrzpnrzjqwghf7zxvfkxq5a6sr65g0gdkv768p83mhsnt0msszapamzx2qvuxqqqqz99gpz55yqqqqqqqqqqqqqq9qrzjq25carzepgd4vqsyn44jrk85ezrpju92xyrk9apw4cdjh6yrwt5jgqqqqz99gpz55yqqqqqqqqqqqqqq9qsp5070fsrfdknwpv2rfeju44k37uj362xcrzmey4se70kpzusy6c0pq9qxpqysgqcj990yc9lg5wje9u6myceakzff95q8qdwrslx69xh526y764dwlxt4v90qhwtmesukfaute6hkh9952t5f5gqjmrry3c6y0k7y2ltkcphyxfp2',
 'preimage': '9de11f4b1bbaf9c18b8b28ee0747fdd64d124b7ee05090f4edc377e73393c2dc',
 'amount': 1,
 'currency': 'usd',
 'payment_method': 'lightning',
 'title': '1 Credit Package',
 'description': 'Purchase 1 credit for API access',
 'type': 'top-up'}

### AI Agent Integration

We will show how to enable your AI assistant to handle payments using [Claudette](https://claudette.answer.ai), Answer.ai convenient wrapper for Claude. You'll need to export your `ANTHROPIC_API_KEY` as env variable for this to work.


In [ ]:
from claudette import Chat, models

In [ ]:
import os
# os.environ['ANTHROPIC_LOG'] = 'debug'

To print every HTTP request and response in full, uncomment the above line.

In [ ]:
fs.balance()

[{'id': 15, 'balance': 6470, 'currency': 'usd'}]

In [ ]:

chat = Chat(models[1], sp='You are a helpful assistant that can pay offers.', tools=fs.as_tools())
pr = f"Could you pay the cheapest offer using lightning {ofs}?"
r = chat.toolloop(pr, trace_func=print)
r

Message(id='msg_01SmWra463n6Q5wivTQS41B9', content=[TextBlock(text="Certainly! I can help you pay for the cheapest offer using Lightning. Based on the information you've provided, there's only one offer available, so we'll proceed with that one. Let's use the `pay` function to complete this transaction.\n\nFirst, let's organize the information we have:\n\n1. There's one offer for 1 credit at $0.01 (1 cent).\n2. The payment method is Lightning.\n3. We have a payment context token and a payment request URL.\n\nNow, let's call the `pay` function with the required parameters:", type='text'), ToolUseBlock(id='toolu_016nRMYdemRYBEuVq8JkWPe4', input={'purl': 'https://stock.l402.org/l402/payment-request', 'pct': 'edb53dec-28f5-4cbb-924a-20e9003c20e1', 'amount': 1, 'balance': 1, 'currency': 'USD', 'description': 'Purchase 1 credit for API access', 'offer_id': 'offer_c668e0c0', 'payment_methods': ['lightning'], 'title': '1 Credit Package', 'type': 'top-up'}, name='pay', type='tool_use')], model=

Great news! The payment has been successfully processed. Here's a summary of the transaction:

1. Payment Status: Success
2. Amount Paid: 1 cent (USD)
3. Payment Method: Lightning
4. Title: 1 Credit Package
5. Description: Purchase 1 credit for API access
6. Type: Top-up
7. Transaction ID: 241
8. Created At: 2024-12-26T10:24:13.845Z

The payment has been completed successfully, and you should now have 1 credit for API access added to your account. Is there anything else you would like to know about this transaction or any other assistance you need?

<details>

- id: `msg_011KopqDeH4CNLt8bKTLyGQD`
- content: `[{'text': "Great news! The payment has been successfully processed. Here's a summary of the transaction:\n\n1. Payment Status: Success\n2. Amount Paid: 1 cent (USD)\n3. Payment Method: Lightning\n4. Title: 1 Credit Package\n5. Description: Purchase 1 credit for API access\n6. Type: Top-up\n7. Transaction ID: 241\n8. Created At: 2024-12-26T10:24:13.845Z\n\nThe payment has been completed successfully, and you should now have 1 credit for API access added to your account. Is there anything else you would like to know about this transaction or any other assistance you need?", 'type': 'text'}]`
- model: `claude-3-5-sonnet-20240620`
- role: `assistant`
- stop_reason: `end_turn`
- stop_sequence: `None`
- type: `message`
- usage: `{'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'input_tokens': 1942, 'output_tokens': 155}`

</details>

In [ ]:
fs.balance()

[{'id': 15, 'balance': 6469, 'currency': 'usd'}]